# Case 6 — Reconstruction loss x cyclic conditioning

**Reproduces:** Fig 4.14

Point anomalies, point mode. Plain MLP-VAE with MSE amplifies an isolated spike into a high score. Once the model is also conditioned on cyclic time features (a tight expectation of 'normal' at this exact hour/day), the thesis found MAE's linear penalty stops amplifying deviations the same way MSE does — the anomaly-score distribution flattens out and separation from normal samples gets worse, not better.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the small synthetic ERA5-shaped dataset shipped with the repo (`scripts/generate_mini_era5.py`) — no data download, no license issues. Numbers will differ from the thesis's real-ERA5 figures (much smaller warmup/test period, noisier), but the *qualitative* effect described above should still show up.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# TODO: update this URL once the repo is pushed to GitHub
REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Generates data/era5/*.csv — fully synthetic, ERA5-shaped, no download needed
!python scripts/generate_mini_era5.py

## Run the suite

`notebooks/cases/case06_reconstruction_loss_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially on the one Colab GPU; each is small (mini dataset), so the whole case should finish in a few minutes.

In [ ]:
!python run_regression.py notebooks/cases/case06_reconstruction_loss_suite.yaml \
    --session runs/regression/case06_reconstruction_loss --gpus 0

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py runs/regression/case06_reconstruction_loss

In [ ]:
import pandas as pd
perf = pd.read_csv("runs/regression/case06_reconstruction_loss/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

In [ ]:
# Display the key comparison plot(s) inline
import glob
from IPython.display import Image, display

print("Score histograms — compare how spread out the anomaly (red) distribution is:")
for p in sorted(glob.glob("runs/regression/case06_reconstruction_loss/cross_compare/point/score_hist_MLP.png")):
    display(Image(filename=p))